# FedGATSage — Kaggle Run Notebook
**Graph-based Federated Learning for IoT Intrusion Detection**  
Paper: [Scientific Reports (2025)](https://doi.org/10.1038/s41598-025-25175-1)

This notebook downloads the datasets, installs dependencies, and runs the
experiment by calling the project scripts directly.

> **Prerequisites**: push the full repo (`src/`, `experiments/`, `preprocess_data.py`,
> `fix-NF-TON-IoT-dataset.py`) to your Kaggle notebook via *Add data → Upload*
> or attach it as a dataset.

## 0 — Setup: clone repository & install dependencies
Source: [https://github.com/ioget/FedGATSage-Visiting-code](https://github.com/ioget/FedGATSage-Visiting-code)

In [ ]:
import os

REPO_URL = 'https://github.com/ioget/FedGATSage-Visiting-code'
REPO_DIR = 'FedGATSage-Visiting-code'

# Clone repo if not already present
if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_URL} ...')
    os.system(f'git clone {REPO_URL}')
else:
    print(f'{REPO_DIR} already cloned — pulling latest changes ...')
    os.system(f'git -C {REPO_DIR} pull')

# Move into repo root so all script paths resolve correctly
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
# Install all dependencies via the project install script
import subprocess, sys
subprocess.run([sys.executable, 'fix-install.py'], check=False)

## 2 — Download datasets from Google Drive

In [ ]:
import os

os.makedirs('data', exist_ok=True)

NF_ID  = '18GMQ1Ncz51t1z2oW-2JJT7CiQFAW7Kkk'
CIC_ID = '120rOiEyxrMQZAFH_KlLMMbK85MQvJkKi'

if not os.path.exists('data/NF-ToN-IoT.csv'):
    print("Downloading NF-ToN-IoT...")
    !gdown {NF_ID} -O data/NF-ToN-IoT.csv
else:
    print("NF-ToN-IoT.csv already present.")

if not os.path.exists('data/CIC-ToN-IoT.csv'):
    print("Downloading CIC-ToN-IoT...")
    !gdown {CIC_ID} -O data/CIC-ToN-IoT.csv
else:
    print("CIC-ToN-IoT.csv already present.")

!ls -lh data/

## 3 — Fix NF-ToN-IoT column schema

In [ ]:
# Renames NF columns to match CIC schema expected by the code
# (IPV4_SRC_ADDR -> Src IP, etc.) and derives missing features
if not os.path.exists('data/NF-ToN-IoT-fixed.csv'):
    !python fix-NF-TON-IoT-dataset.py
else:
    print("NF-ToN-IoT-fixed.csv already exists, skipping fix.")

## 4 — Preprocess: split into federated client files
Creates `data/cic/` and `data/nf/` each containing
`client_1.csv` … `client_5.csv` + `test.csv`.

In [ ]:
# CIC-ToN-IoT
!python preprocess_data.py \
    --input_file data/CIC-ToN-IoT.csv \
    --output_dir data/cic \
    --num_clients 5 \
    --seed 42

In [ ]:
# NF-ToN-IoT (uses the fixed file)
!python preprocess_data.py \
    --input_file data/NF-ToN-IoT-fixed.csv \
    --output_dir data/nf \
    --num_clients 5 \
    --seed 42

## 5 — Run experiment: CIC-ToN-IoT
Change `--num_rounds` or remove `--demo_mode` for the full run.

In [ ]:
!python experiments/fedgatsage_experiment.py \
    --data_dir data/cic \
    --dataset cic_ton_iot \
    --num_clients 5 \
    --num_rounds 5 \
    --detector_types temporal content behavioral \
    --device cpu \
    --output_dir results/cic \
    --seed 42 \
    --demo_mode

## 6 — Run experiment: NF-ToN-IoT

In [ ]:
!python experiments/fedgatsage_experiment.py \
    --data_dir data/nf \
    --dataset nf_ton_iot \
    --num_clients 5 \
    --num_rounds 5 \
    --detector_types temporal content behavioral \
    --device cpu \
    --output_dir results/nf \
    --seed 42 \
    --demo_mode

## 7 — Show results

In [ ]:
import json, glob

for result_file in sorted(glob.glob('results/**/*.json', recursive=True)):
    print(f"\n{'='*60}")
    print(f"Results: {result_file}")
    print('='*60)
    with open(result_file) as f:
        data = json.load(f)
    final = data.get('final_results', {}).get('evaluation', {})
    if final:
        print(f"  Accuracy         : {final.get('accuracy', 'N/A')}")
        print(f"  Balanced Accuracy: {final.get('balanced_accuracy', 'N/A')}")
        print(f"  Macro F1         : {final.get('macro_f1', 'N/A')}")
        print(f"  Weighted F1      : {final.get('weighted_f1', 'N/A')}")
        if 'per_class_detailed' in final:
            print("\n  Per-class F1:")
            for cls, m in final['per_class_detailed'].items():
                print(f"    {cls:<15} F1={m['f1']:.4f}  P={m['precision']:.4f}  R={m['recall']:.4f}")
    else:
        print('  No evaluation results found.')

In [ ]:
# Show saved plots
from IPython.display import Image, display
import glob

for img in sorted(glob.glob('results/**/*.png', recursive=True)):
    print(img)
    display(Image(img))